In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver # Every State values is saved on RAM (Once the laptop is shutdown, all the values will be deleted). We use this for DEMO. Not for Production ready.

In [2]:
import os
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
llm = ChatOpenAI(
    model = "openai/gpt-oss-20b",
    api_key = os.getenv("GROK_API_KEY"),
    base_url = "https://api.groq.com/openai/v1",
)

In [5]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation : str

In [6]:
def generate_joke(state : JokeState):
    prompt = f"Generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt).content
    return {
        'joke' : response
    }

In [7]:
def generate_explanation(state : JokeState):
    prompt = f'Write an explanation for the joke - {state['joke']}'
    response = llm.invoke(prompt).content
    return {
        'explanation' : response
    }

In [8]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

In [10]:
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer = checkpointer) # We are saving every State value with the help of InMemorySaver

* If we pass another topic as "pasta" and with thread_id as "2". Previous thread_id "1" execution would also be saved. As we are using InMemorySaver, if we ShutDown our PC than the history would be gone.

In [11]:
config1 = {
    'configurable' : {"thread_id" : "1"}
}
workflow.invoke(
    {
        "topic" : "pizza"
    },
    config = config1
)

{'topic': 'pizza',
 'joke': 'Why did the pizza bring a ladder to the party?  \nBecause it heard the toppings were going to be *above* the rest of the food!',
 'explanation': '**Why the joke works – a quick unpacking**\n\n| Element | What it means | How it contributes to the humor |\n|---------|----------------|----------------------------------|\n| **“Pizza”** | A food that is literally flat and has toppings on top of it. | It’s the absurd “character” that can go to a party and bring a ladder. |\n| **“Ladder”** | A tool people use to reach something that’s physically higher. | The idea of a pizza carrying a ladder is ridiculous—nothing about a pizza normally needs climbing gear. |\n| **“Toppings”** | The ingredients (pepperoni, cheese, veggies) that are *on top* of a pizza. | The pun starts here: “toppings” are literally above the crust. |\n| **“Above”** | (a) “at a higher physical level” <br> (b) “better or superior” (e.g., “he’s above the rest”). | The joke plays on both senses. The 

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza bring a ladder to the party?  \nBecause it heard the toppings were going to be *above* the rest of the food!', 'explanation': '**Why the joke works – a quick unpacking**\n\n| Element | What it means | How it contributes to the humor |\n|---------|----------------|----------------------------------|\n| **“Pizza”** | A food that is literally flat and has toppings on top of it. | It’s the absurd “character” that can go to a party and bring a ladder. |\n| **“Ladder”** | A tool people use to reach something that’s physically higher. | The idea of a pizza carrying a ladder is ridiculous—nothing about a pizza normally needs climbing gear. |\n| **“Toppings”** | The ingredients (pepperoni, cheese, veggies) that are *on top* of a pizza. | The pun starts here: “toppings” are literally above the crust. |\n| **“Above”** | (a) “at a higher physical level” <br> (b) “better or superior” (e.g., “he’s above the rest”). | The joke plays o

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza bring a ladder to the party?  \nBecause it heard the toppings were going to be *above* the rest of the food!', 'explanation': '**Why the joke works – a quick unpacking**\n\n| Element | What it means | How it contributes to the humor |\n|---------|----------------|----------------------------------|\n| **“Pizza”** | A food that is literally flat and has toppings on top of it. | It’s the absurd “character” that can go to a party and bring a ladder. |\n| **“Ladder”** | A tool people use to reach something that’s physically higher. | The idea of a pizza carrying a ladder is ridiculous—nothing about a pizza normally needs climbing gear. |\n| **“Toppings”** | The ingredients (pepperoni, cheese, veggies) that are *on top* of a pizza. | The pun starts here: “toppings” are literally above the crust. |\n| **“Above”** | (a) “at a higher physical level” <br> (b) “better or superior” (e.g., “he’s above the rest”). | The joke plays 

### Time Travel

In [14]:
workflow.get_state(
    {
        "configurable" : {
            "thread_id" : 1,
            "checkpoint_id" : "1f1e64c2-13e6-6fa9-8000-b6e4eccb79d8"
        }
    }
)

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1e64c2-13e6-6fa9-8000-b6e4eccb79d8'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-11-22T08:10:07.483895+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e64c2-13e2-6175-bfff-36d2dd9e07d6'}}, tasks=(PregelTask(id='8a38749c-8511-c9e0-8b82-2fcbcebc87f4', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza bring a ladder to the party?  \nBecause it heard the toppings were going to be *above* the rest of the food!'}),), interrupts=())

In [15]:
workflow.invoke(None,
                {
                    "configurable" : {
                                "thread_id" : 1,
                                "checkpoint_id" : "1f1e64c2-13e6-6fa9-8000-b6e4eccb79d8"
                            }
                })

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?  \nBecause it wanted to get a little *cheddar* in its dough!',
 'explanation': '**Explanation of the joke**\n\n> *“Why did the pizza apply for a job? Because it wanted to get a little **cheddar** in its dough!”*\n\nAt first glance the sentence seems to be a harmless, whimsical riddle: a pizza, a food item, applying for a job. The humor comes from a double layer of wordplay that turns a simple statement into a clever pun.\n\n---\n\n### 1. The two key puns\n\n| Word in the joke | Common meaning | Slang / alternative meaning |\n|-------------------|----------------|------------------------------|\n| **Cheddar**       | A type of cheese (often used on pizza) | Slang for “money” (e.g., “I need some cheddar”) |\n| **Dough**         | The liquid mixture used to make pizza dough | Slang for “money” (e.g., “He’s got a lot of dough”) |\n\nBecause both *cheddar* and *dough* have these dual meanings, the joke links them together in a 

In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \nBecause it wanted to get a little *cheddar* in its dough!', 'explanation': '**Explanation of the joke**\n\n> *“Why did the pizza apply for a job? Because it wanted to get a little **cheddar** in its dough!”*\n\nAt first glance the sentence seems to be a harmless, whimsical riddle: a pizza, a food item, applying for a job. The humor comes from a double layer of wordplay that turns a simple statement into a clever pun.\n\n---\n\n### 1. The two key puns\n\n| Word in the joke | Common meaning | Slang / alternative meaning |\n|-------------------|----------------|------------------------------|\n| **Cheddar**       | A type of cheese (often used on pizza) | Slang for “money” (e.g., “I need some cheddar”) |\n| **Dough**         | The liquid mixture used to make pizza dough | Slang for “money” (e.g., “He’s got a lot of dough”) |\n\nBecause both *cheddar* and *dough* have these dual meanings, the joke links